# ATML PA4 · Task 3 — Policy distillation and speculative decoding
**CS-6304 / EE-5102** · code: https://github.com/danishrasool10/ATML-PA-4 (`task_3/`, shared code in `common/`)

### Before running
1. **Settings → Accelerator: GPU T4 ×2**, **Internet: On**.
2. **Add-ons → Secrets**: add `HF_TOKEN` (Hugging Face read token, notebook access on). Optional `GH_TOKEN` (GitHub token; only for a private repo or `PUSH_RESULTS`).
3. Run the cells top to bottom, or **Save Version → Save & Run All (Commit)** to run unattended.

### What happens
`python -m pa4t3.run_all` runs unit tests, `setup_data.py --public` and `--prefix-banks`, the Step 1 validation gate (chi-square on 100,000 Markov-toy sequences, KV rollback, merged-vs-unmerged logits; sweeps refuse to start if it fails), the four distillation runs (off-/on-policy x seeds 6304, 6305) on the two GPUs, then prefix-bank alignment and standalone quality; afterwards, alone on an idle machine: pretrained-draft and merged-student speculative benchmarks (gamma 2 and 4), fixed-64-token throughput and the analysis with deployment break-even. Outputs: `task_3/results/`.

GPU use: two lanes for validation / distillation / banks / standalone, then one quiet GPU for the speculative-decoding benchmarks. FP16 everywhere (T4 has no bf16). Everything is resumable: finished jobs are skipped when you run the main cell again. The runner stops
launching jobs after `DEADLINE_HOURS` (Kaggle's limit is 12 h); to continue, **Save Version**, add that output as an *Input* and set `RESTORE_DIR`.

In [ ]:
import os, subprocess, sys, shutil, json, time
import torch
n = torch.cuda.device_count()
print("GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
assert n >= 1, "turn on a GPU accelerator (T4 x2) in the notebook settings"

## 1. Settings

In [ ]:
TASK = 3
MODULE = "pa4t3"
WORK = "/kaggle/working"
CODE_URL = "https://github.com/danishrasool10/ATML-PA-4.git"
CODE_REF = "main"
DATA_URL = "https://github.com/AbDu11aHHH/atml-pa4-datasets.git"
GPUS = ",".join(str(i) for i in range(min(2, n)))   # "0,1"
DEADLINE_HOURS = 11.0        # stop launching jobs after this; running jobs get a 15 min grace
RESTORE_DIR = ""             # e.g. "/kaggle/input/<previous-output>/ATML-PA-4/task_3/results" to continue an earlier session
RUN_SMOKE = False            # True: first run this task on tiny models (CPU plumbing check, no real results)
PUSH_RESULTS = False         # True: commit task_3/results to main (needs a GH_TOKEN secret with write access)
REPO = f"{WORK}/ATML-PA-4"
DATA = f"{WORK}/atml-pa4-datasets"
TASK_DIR = f"{REPO}/task_3"

## 2. Secrets, code, dependencies

In [ ]:
secrets = {}
try:
    from kaggle_secrets import UserSecretsClient
    us = UserSecretsClient()
    for k in ("HF_TOKEN", "GH_TOKEN"):
        try:
            secrets[k] = us.get_secret(k)
        except Exception:
            print(f"secret {k}: not set")
except ImportError:
    secrets = {k: os.environ[k] for k in ("HF_TOKEN", "GH_TOKEN") if k in os.environ}
if "HF_TOKEN" in secrets:
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = secrets["HF_TOKEN"]
print("HF token set:", "HF_TOKEN" in secrets)

def clone(url, dest, ref=None):
    auth = url.replace("https://", f"https://x-access-token:{secrets['GH_TOKEN']}@") if "GH_TOKEN" in secrets else url
    if os.path.isdir(dest + "/.git"):
        subprocess.run(["git", "-C", dest, "fetch", "-q", auth] + ([ref] if ref else []), check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", "-B", ref or "main", "FETCH_HEAD"], check=True)
    else:
        subprocess.run(["git", "clone", "-q"] + (["-b", ref] if ref else []) + [auth, dest], check=True)
        subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url], check=True)   # keep the token out of .git/config

clone(CODE_URL, REPO, CODE_REF)
clone(DATA_URL, DATA)
print(subprocess.run(f"git -C {REPO} log --oneline -1; git -C {DATA} log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# keep Kaggle's CUDA torch; install the rest (pins transformers==4.57.1, huggingface_hub==0.36.2)
tv = subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__.split('+')[0])"], capture_output=True, text=True).stdout.strip()
open("/tmp/constraints.txt", "w").write(f"torch=={tv}\n")
r = subprocess.run(f"pip install -q -c /tmp/constraints.txt -r {REPO}/requirements.txt -r {DATA}/requirements-public.txt", shell=True, capture_output=True, text=True)
print(r.stdout[-1200:], r.stderr[-1200:])
print(subprocess.run([sys.executable, "-c", "import torch,transformers,huggingface_hub as h;print('torch',torch.__version__,'transformers',transformers.__version__,'hub',h.__version__)"], capture_output=True, text=True).stdout)
from huggingface_hub import login
if "HF_TOKEN" in secrets:
    login(token=secrets["HF_TOKEN"], add_to_git_credential=False)
os.environ["PA4_DATA_ROOT"] = DATA
os.environ["PYTHONPATH"] = f"{REPO}/common:{TASK_DIR}/src"
sys.path[:0] = [f"{REPO}/common", f"{TASK_DIR}/src"]
os.makedirs(f"{TASK_DIR}/results", exist_ok=True)
if RESTORE_DIR and os.path.isdir(RESTORE_DIR):
    shutil.copytree(RESTORE_DIR, f"{TASK_DIR}/results", dirs_exist_ok=True)
    print("restored", RESTORE_DIR)

## 3. Optional: plumbing check on tiny models (CPU, no real results)

In [ ]:
if RUN_SMOKE:
    !cd {TASK_DIR} && python -u -m {MODULE}.run_all --smoke --gpus ""

## 4. Run the task

In [ ]:
!cd {TASK_DIR} && python -u -m {MODULE}.run_all --gpus {GPUS} --deadline-hours {DEADLINE_HOURS}

## 5. What finished / what failed

In [ ]:
p = f"{TASK_DIR}/results/.jobs/summary.json"
if os.path.exists(p):
    s = json.load(open(p))
    for k, v in s.items():
        print(f"{v:12s} {k}")
    todo = [k for k, v in s.items() if v != "done"]
    print("\nNOT DONE:", todo or "nothing")
    for k in todo:
        lf = f"{TASK_DIR}/results/logs/{k}.log"
        if os.path.exists(lf):
            print(f"\n--- {k} (last lines) ---\n" + "".join(open(lf).readlines()[-15:]))

## 6. Package the results
Writes `/kaggle/working/pa4_task3_results.zip`; press *Save Version* afterwards and download it from the Output tab.

In [ ]:
os.chdir(WORK)
shutil.make_archive(f"{WORK}/pa4_task{TASK}_results", "zip", TASK_DIR, "results")
print(subprocess.run(f"du -sh {WORK}/pa4_task{TASK}_results.zip; du -sh {TASK_DIR}/results/*/ 2>/dev/null | sort -h | tail", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# optional: commit the results to main (like ATML-PA3 keeps task_N/results in the repo). Needs GH_TOKEN with write access.
if PUSH_RESULTS:
    assert "GH_TOKEN" in secrets, "add a GH_TOKEN secret first"
    auth = CODE_URL.replace("https://", f"https://x-access-token:{secrets['GH_TOKEN']}@")
    sh = (f"cd {REPO} && git add -f task_{TASK}/results && git -c user.name=kaggle-run -c user.email=noreply@example.com "
          f"commit -qm 'Task {TASK}: Kaggle 2xT4 results' && git pull -q --rebase {auth} main && git push -q {auth} HEAD:main")
    r = subprocess.run(sh, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(secrets["GH_TOKEN"], "***")[-800:])